In [38]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict,Annotated
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from pydantic import BaseModel,Field
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage,BaseMessage

load_dotenv()

True

In [39]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [40]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile"   # You can change model here
)
def chat_node(state:ChatState):

    massages = state['messages']

    response=llm.invoke(massages)

    return {'messages': response}



In [41]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chat_bot=graph.compile(checkpointer=checkpointer)

In [42]:
# initial_state = {
#     'messages': [HumanMessage(content="Capital of India?")]
# }

# chat_bot.invoke(initial_state)[ 'messages'] [-1].content

In [43]:
thread_id ='1'

while True:
    user_message = input('Type here: ')
    print(f'User: {user_message}')

    if user_message.lower() in ['exit', 'quit']:
        print("Exiting chat.")
        break
    
    config = {'configurable':{
        'thread_id': thread_id
    }}

    response = chat_bot.invoke({'messages': [HumanMessage(content=user_message)]}, config=config)
    print(f'Bot: {response["messages"][-1].content}')

User: i am vipul
Bot: Hello Vipul, it's nice to meet you. Is there something I can help you with or would you like to chat?
User: what is my name
Bot: Your name is Vipul.
User: exit
Exiting chat.
